# Lấy Google Trends Data & Re-normalize Daily Batches

## Mục tiêu
1. Gọi Google Trends API lấy data cho `Elden Ring` và `Ready Or Not` (01/12/2021 → 31/03/2026)
2. Google trả về **weekly** data (vì timeframe > 5 tháng). Ta tổng hợp thành **monthly reference**
3. Lưu cả weekly raw và monthly aggregated vào `data/raw/`
4. Dùng monthly reference mới để re-normalize daily batches từ `*_batches.csv`

## Cấu hình API
- **Location**: Worldwide (`geo=''`)
- **Search type**: Web Search (`gprop=''`)
- **Timeframe**: `2021-12-01 2026-03-31`

## Tại sao Google không trả daily?
Google Trends tự chọn granularity theo độ dài timeframe:
- < 1 tuần → hourly
- 1 tuần – ~5 tháng → **daily**
- ~5 tháng – ~5 năm → **weekly** ← trường hợp của ta
- \> 5 năm → monthly

Không thể ép daily cho khoảng > 5 tháng trong 1 lần query. Vì vậy notebook `gg_trends_daily.ipynb` chia batch 1 tháng/lần để lấy daily, rồi notebook này cung cấp monthly reference để rescale.

## Cell 1: Setup

In [ ]:
!pip install pytrends -q

import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = os.path.join('..', 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
print(f'Output directory: {os.path.abspath(RAW_DIR)}')

## Cell 2: Cấu hình

In [ ]:
TIMEFRAME = '2021-12-01 2026-03-31'

GAMES = {
    'EldenRing': {
        'keyword': 'Elden Ring',
        'weekly_output': 'EldenRing_trend_weekly.csv',
        'monthly_output': 'EldenRing_trend_monthly_past5years.csv',
        'batches_file': 'EldenRing_batches.csv',
        'daily_output': 'EldenRing_trend_daily.csv'
    },
    'ReadyOrNot': {
        'keyword': 'Ready Or Not',
        'weekly_output': 'ReadyOrNot_trend_weekly.csv',
        'monthly_output': 'ReadyOrNot_trend_monthly_past5years.csv',
        'batches_file': 'ReadyOrNot_batches.csv',
        'daily_output': 'ReadyOrNot_trend_daily.csv'
    }
}

print(f'Timeframe: {TIMEFRAME}')
print(f'Games: {list(GAMES.keys())}')

## Cell 3: Lấy Weekly Data từ Google Trends & Tổng hợp thành Monthly

- Google Trends trả **weekly** cho timeframe ~4 năm
- Ta tổng hợp weekly → monthly bằng cách lấy **trung bình (mean)** các tuần trong tháng
- Sau đó **chuẩn hóa** monthly values về thang 0–100 (giá trị cao nhất = 100)

In [ ]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl='en-US', tz=420)

WAIT_BETWEEN_GAMES = 30

for idx, (game_name, cfg) in enumerate(GAMES.items()):
    keyword = cfg['keyword']
    
    print(f"\n{'='*60}")
    print(f"  Fetching: '{keyword}'")
    print(f"  Timeframe: {TIMEFRAME}")
    print(f"  geo='', gprop='' (Worldwide, Web Search)")
    print(f"{'='*60}")
    
    try:
        pytrends.build_payload([keyword], timeframe=TIMEFRAME, geo='', gprop='')
        df = pytrends.interest_over_time()
        
        if df.empty:
            print('  ⚠️ Không có dữ liệu!')
            continue
        
        if 'isPartial' in df.columns:
            df = df.drop(columns=['isPartial'])
        
        # --- Lưu weekly raw ---
        weekly = pd.DataFrame({
            'Date': df.index,
            'Weekly_Value': df[keyword].values
        }).sort_values('Date').reset_index(drop=True)
        
        weekly_path = os.path.join(RAW_DIR, cfg['weekly_output'])
        weekly.to_csv(weekly_path, index=False)
        print(f'  ✅ Weekly saved: {weekly_path} ({len(weekly)} rows)')
        
        # --- Tổng hợp weekly → monthly ---
        weekly['Date'] = pd.to_datetime(weekly['Date'])
        weekly['YearMonth'] = weekly['Date'].dt.to_period('M')
        
        monthly = weekly.groupby('YearMonth')['Weekly_Value'].mean().reset_index()
        monthly.columns = ['YearMonth', 'Monthly_Value']
        
        # Chuẩn hóa về thang 0-100
        max_val = monthly['Monthly_Value'].max()
        monthly['Monthly_Value'] = (monthly['Monthly_Value'] / max_val * 100).round(2)
        
        # Tạo cột Date (ngày đầu tháng)
        monthly['Date'] = monthly['YearMonth'].dt.to_timestamp()
        monthly = monthly[['Date', 'Monthly_Value']].sort_values('Date').reset_index(drop=True)
        
        monthly_path = os.path.join(RAW_DIR, cfg['monthly_output'])
        monthly.to_csv(monthly_path, index=False)
        print(f'  ✅ Monthly saved: {monthly_path} ({len(monthly)} months)')
        print(f'     Range: {monthly["Date"].min()} → {monthly["Date"].max()}')
        print(f'     Min: {monthly["Monthly_Value"].min()}, Max: {monthly["Monthly_Value"].max()}')
        print()
        print('  --- Monthly Data ---')
        print(monthly.to_string(index=False))
        
    except Exception as e:
        print(f'  ❌ Lỗi: {e}')
        print(f'     → Chờ vài phút rồi chạy lại cell này.')
    
    if idx < len(GAMES) - 1:
        print(f'\n  ⏳ Chờ {WAIT_BETWEEN_GAMES}s...')
        time.sleep(WAIT_BETWEEN_GAMES)

print(f'\n🎉 Hoàn thành!')

## Cell 4: Re-normalize Daily Batches

- Đọc `*_batches.csv` (daily raw, thang 0–100 riêng mỗi tháng)
- Đọc `*_trend_monthly_past5years.csv` (monthly reference mới, thang 0–100 toàn cục)
- Công thức: `Trend_Value = batch_value × (monthly_ref / 100)`
- Export ra `*_trend_daily.csv` (ghi đè file cũ)

In [ ]:
for game_name, cfg in GAMES.items():
    print(f"\n{'='*60}")
    print(f"  Re-normalizing: {game_name}")
    print(f"{'='*60}")
    
    # 1. Load daily batches
    batches_path = os.path.join(RAW_DIR, cfg['batches_file'])
    if not os.path.exists(batches_path):
        print(f'  ⚠️ Không tìm thấy {batches_path}.')
        print(f'     Chạy gg_trends_daily.ipynb Cell 3 trước để lấy daily batches.')
        continue
    
    df_raw = pd.read_csv(batches_path, parse_dates=['date'])
    df_raw = df_raw.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
    print(f'  Daily batches: {len(df_raw)} rows')
    
    # 2. Load monthly reference (mới)
    monthly_path = os.path.join(RAW_DIR, cfg['monthly_output'])
    if not os.path.exists(monthly_path):
        print(f'  ⚠️ Không tìm thấy {monthly_path}. Chạy Cell 3 trước.')
        continue
    
    df_monthly = pd.read_csv(monthly_path, parse_dates=['Date'])
    df_monthly['month'] = df_monthly['Date'].dt.to_period('M').astype(str)
    monthly_map = dict(zip(df_monthly['month'], df_monthly['Monthly_Value']))
    print(f'  Monthly reference: {len(df_monthly)} months')
    
    # 3. Rescale
    df_raw['month'] = df_raw['date'].dt.to_period('M').astype(str)
    df_raw['monthly_ref'] = df_raw['month'].map(monthly_map)
    
    missing = df_raw['monthly_ref'].isna().sum()
    if missing > 0:
        print(f'  ⚠️ {missing} ngày không có monthly ref → dùng ffill/bfill')
        df_raw['monthly_ref'] = df_raw['monthly_ref'].ffill().bfill()
    
    df_raw['scaled'] = df_raw['value'] * (df_raw['monthly_ref'] / 100.0)
    
    # 4. Export
    export = pd.DataFrame({
        'Date': df_raw['date'],
        'Trend_Value': df_raw['scaled']
    }).sort_values('Date').reset_index(drop=True)
    
    export['Trend_Value'] = export['Trend_Value'].interpolate('linear').fillna(0).clip(lower=0)
    
    out_path = os.path.join(RAW_DIR, cfg['daily_output'])
    export.to_csv(out_path, index=False)
    
    print(f'  ✅ Saved: {out_path}')
    print(f'     Rows: {len(export)}')
    print(f'     Range: {export["Date"].min()} → {export["Date"].max()}')
    print(f'     Trend_Value: min={export["Trend_Value"].min():.2f}, max={export["Trend_Value"].max():.2f}')
    print()
    print('  --- 5 dòng đầu ---')
    print(export.head())

print(f'\n🎉 Re-normalize hoàn thành!')

## Cell 5: Validation Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for idx, (game_name, cfg) in enumerate(GAMES.items()):
    ax = axes[idx]
    
    # Weekly raw
    weekly_path = os.path.join(RAW_DIR, cfg['weekly_output'])
    if os.path.exists(weekly_path):
        dw = pd.read_csv(weekly_path, parse_dates=['Date'])
        ax.plot(dw['Date'], dw['Weekly_Value'], 'g^-', label='Weekly (raw from API)', markersize=3, lw=1, alpha=0.5)
    
    # Monthly aggregated
    monthly_path = os.path.join(RAW_DIR, cfg['monthly_output'])
    if os.path.exists(monthly_path):
        dm = pd.read_csv(monthly_path, parse_dates=['Date'])
        ax.plot(dm['Date'], dm['Monthly_Value'], 'ro-', label='Monthly (aggregated)', markersize=5, lw=2)
    
    # Daily rescaled
    daily_path = os.path.join(RAW_DIR, cfg['daily_output'])
    if os.path.exists(daily_path):
        dd = pd.read_csv(daily_path, parse_dates=['Date'])
        ax.plot(dd['Date'], dd['Trend_Value'], 'b-', label='Daily (rescaled)', alpha=0.6, lw=0.8)
    
    ax.set_title(f'{game_name}: Weekly → Monthly → Daily Rescaled', fontsize=14, fontweight='bold')
    ax.set_ylabel('Search Interest')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(os.path.join('..', 'reports', 'figures'), exist_ok=True)
plt.savefig(os.path.join('..', 'reports', 'figures', 'trends_daily_vs_monthly_past5years.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/figures/trends_daily_vs_monthly_past5years.png')

## Hướng dẫn sử dụng

1. **Cell 1–2**: Setup & cấu hình
2. **Cell 3**: Lấy weekly từ API → tổng hợp thành monthly. Nếu bị 429, chờ vài phút.
3. **Cell 4**: Re-normalize daily batches (cần đã có `*_batches.csv` từ `gg_trends_daily.ipynb`)
4. **Cell 5**: Vẽ biểu đồ kiểm tra

### Output files:
- `*_trend_weekly.csv` — Weekly raw từ API
- `*_trend_monthly_past5years.csv` — Monthly aggregated (mean, chuẩn hóa 0–100)
- `*_trend_daily.csv` — Daily đã rescale (ghi đè file cũ)

### Bước tiếp theo:
Chạy `python src\data\process_and_merge.py` để tạo lại `*_Final_Merged.csv`.